In [ ]:
import pandas as pd
import numpy as np
import re
import string
import pickle

In [2]:
import string

def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, ' ')
    return text

In [3]:
import pickle

with open('../static/model/model.pickle','rb') as f:
    model=pickle.load(f)

In [4]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines()

In [5]:
vocab=pd.read_csv('../static/model/vocabulary.txt',header=None)
tokens=vocab[0].tolist()

In [6]:
from nltk.stem import PorterStemmer
ps=PorterStemmer()

In [7]:

def preprocessing(text):
    data = pd.DataFrame([text], columns=['tweet'])

    data["tweet"] = data["tweet"].apply(
        lambda x: " ".join(x.lower() for x in x.split())
    )

    data["tweet"] = data["tweet"].apply(
        lambda x: " ".join(
            re.sub(r'^https?:\/\/.*[\r\n]*', '', x, flags=re.MULTILINE)
            for x in x.split()
        )
    )

    data["tweet"] = data["tweet"].apply(remove_punctuations)

    data["tweet"] = data["tweet"].str.replace(r'\d+', '', regex=True)

    data["tweet"] = data["tweet"].apply(
        lambda x: " ".join(x for x in x.split() if x not in sw)
    )

    data["tweet"] = data["tweet"].apply(
        lambda x: " ".join(ps.stem(x) for x in x.split())
    )

    return data["tweet"]

In [8]:
def vectorizer(ds, vocabulary):
    vectorized_lst = []

    for sentence in ds:
        sentence_lst = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_lst[i] = 1

        vectorized_lst.append(sentence_lst)

    vectorized_lst_new = np.asarray(vectorized_lst, dtype=np.float32)

    return vectorized_lst_new

In [9]:
def get_prediction(vectorized_text):
    prediction=model.predict(vectorized_text)
    if prediction==1:
        return 'negative'
    else:
        return 'positive'

In [27]:
import re
txt="I regret buying this phone."
preprocessed_txt=preprocessing(txt)
vectorized_txt=vectorizer(preprocessed_txt,tokens)
prediction=get_prediction(vectorized_txt)
prediction

'negative'

In [14]:
test_words = [
    "good", "excellent", "best", "love", "awesome",
    "bad", "worst", "poor", "hate", "terrible"
]

for word in test_words:
    p = preprocessing(word)
    v = vectorizer(p, tokens)
    print(f"{word:10} -> {get_prediction(v)}")

good       -> negative
excellent  -> negative
best       -> positive
love       -> positive
awesome    -> positive
bad        -> negative
worst      -> negative
poor       -> negative
hate       -> negative
terrible   -> negative


In [15]:
positive_words = [
    "love",
    "awesome",
    "best",
    "great",
    "nice",
    "amazing",
    "wonderful",
    "fantastic",
    "super",
    "perfect",
    "brilliant",
    "excellent",
    "happy",
    "beautiful",
    "liked",
    "favorite",
    "recommend",
    "recommended",
    "impressive",
    "outstanding",
    "delightful",
    "pleasant",
    "valuable",
    "useful",
    "quality",
    "smooth",
    "fast",
    "reliable",
    "satisfied",
    "success"
]

for word in positive_words:
    p = preprocessing(word)
    v = vectorizer(p, tokens)
    print(f"{word:15} -> {get_prediction(v)}")

love            -> positive
awesome         -> positive
best            -> positive
great           -> positive
nice            -> positive
amazing         -> positive
wonderful       -> positive
fantastic       -> negative
super           -> positive
perfect         -> positive
brilliant       -> negative
excellent       -> negative
happy           -> positive
beautiful       -> positive
liked           -> negative
favorite        -> negative
recommend       -> negative
recommended     -> negative
impressive      -> negative
outstanding     -> negative
delightful      -> negative
pleasant        -> negative
valuable        -> negative
useful          -> negative
quality         -> positive
smooth          -> negative
fast            -> positive
reliable        -> negative
satisfied       -> positive
success         -> negative


In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [17]:
data=pd.read_csv('../artifacts/sentiment_analysis.csv')

In [18]:
data[data["tweet"].str.contains("good", case=False, na=False)][["tweet", "label"]]

,tweet,label
9,Photo: #fun #selfie #pool #water #sony #camera...,0
17,"Go crazy !! #iphonesia, #iphone, #instagood, #...",0
40,"If they din’t #create you, don’t let them #des...",0
43,Eeeeee sexy ladies !!!!!! #live #iphone #iphon...,0
57,@apple are you seriously so afraid of your pro...,1
...,...,...
7856,Playing with my new phone #samsung #galaxys4 #...,0
7859,Splash! #goodnight #photo #picture #photograph...,0
7865,"Love my #iphonesia #iphone, #instagood #instag...",0
7897,Slowly moving away from shitty Apple. Goodbye ...,1
